## AlphaFold DB predicted structure for thrombin (UniProt entry name lookup)

`chem.alphafold.download_structures` accepts the same three id forms as
`chem.chembl.download_activities` and `chem.rcsb.download_structures`. It
resolves the id to a UniProt accession and downloads every AlphaFold DB
prediction entry for it (usually one, but very large proteins may be split
into fragments, and some targets without an official prediction have
community-submitted alternatives instead), optionally filtered by a minimum
average pLDDT confidence (`plddt_thres`, 0-100). As with `chem.rcsb`, files
already present in `outdir` are left alone.

In [ ]:
from chem import alphafold

n = alphafold.download_structures(
    "THRB_HUMAN",
    outdir="af_data",
    filetype="pdb",
)
n

### View the predicted structure, colored by pLDDT confidence

In [ ]:
import os

from chem import view3d

pdb_files = sorted(f for f in os.listdir("af_data") if f.endswith(".pdb"))
print(pdb_files)

# AlphaFold stores per-residue pLDDT confidence in the B-factor column.
view3d.render_protein(
    os.path.join("af_data", pdb_files[0]), coloring="bfactor", width=600, height=600
)

### Same structure as a solid volume instead of a cartoon

`view3d.render_protein`'s new `style="surface"` option draws the backbone as a
translucent van der Waals volume (3Dmol.js computes it via marching cubes over a
grid built from atomic radii -- effectively the union of a smooth blob at every
atom, not individual per-atom spheres) instead of a cartoon ribbon. Coloring works
the same way regardless of `style`, so `coloring="bfactor"` still maps pLDDT
confidence onto it.


In [ ]:
# Same file, same pLDDT coloring, but as a solid volume instead of a ribbon.
view3d.render_protein(
    os.path.join("af_data", pdb_files[0]), coloring="bfactor", style="surface", width=600, height=600
)


### Blind pocket detection with chem.protein.list_pockets

This predicted structure has no bound ligand to anchor on, so
`chem.protein.find_pocket` (which picks the pocket nearest a given ligand)
doesn't apply here. `chem.protein.list_pockets` instead runs fpocket and
returns every candidate pocket it detects, sorted by druggability score
descending -- useful for blind pocket detection on apo/predicted structures.


In [ ]:
import pandas as pd

from chem import protein

pockets = protein.list_pockets(os.path.join("af_data", pdb_files[0]))
pockets_df = pd.DataFrame(
    [
        {
            "pocket_id": p["pocket_id"],
            "score": p["score"],
            "druggability_score": p["druggability_score"],
            "volume": p["volume"],
            "n_residues": len(p["residues"]),
        }
        for p in pockets
    ]
)
print(f"{len(pockets_df)} candidate pocket(s) found")
display(pockets_df.style.hide(axis="index").format(precision=3))


### Visualize the druggable pocket candidates

Highlight every pocket with `druggability_score >= 0.1` on the structure, one
color per pocket, on top of a translucent cartoon so the highlighted residues
stand out.


In [ ]:
import itertools

import py3Dmol
from IPython.display import HTML, display

# 3Dmol.js only defines "*Carbon" colorscheme presets for these 8 colors.
_POCKET_COLORS = ["orange", "cyan", "magenta", "yellow", "green", "purple", "blue", "white"]

druggable_pockets = [p for p in pockets if p["druggability_score"] is not None and p["druggability_score"] >= 0.1]
print(f"{len(druggable_pockets)} pocket(s) with druggability_score > 0.2")

with open(os.path.join("af_data", pdb_files[0])) as f:
    pdb_text = f.read()

view = py3Dmol.view(width=700, height=550)
view.addModel(pdb_text, "pdb")
# Translucent backbone so the highlighted pocket residues stand out.
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})

legend_html = ""
for pocket, color in zip(druggable_pockets, itertools.cycle(_POCKET_COLORS)):
    resnums = sorted({r["resnum"] for r in pocket["residues"]})
    view.addStyle({"resi": resnums}, {"stick": {"colorscheme": f"{color}Carbon"}})
    legend_html += (
        f'<div><span style="display:inline-block;width:12px;height:12px;background:{color};border:1px solid #888;margin-right:6px;"></span>'
        f'pocket {pocket["pocket_id"]} (druggability {pocket["druggability_score"]:.3f})</div>'
    )
view.zoomTo()

display(HTML(f'<div style="display:flex; gap:16px;">'
             f'<div id="chem-pockets-view" style="width:700px; height:550px; border:1px solid #ccc;"></div>'
             f'<div>{legend_html}</div></div>'))
view.insert("chem-pockets-view")


### Same pockets, as a filled volume (alpha spheres) instead of residue sticks

`chem.protein.list_pockets`'s `spheres` field is fpocket's own alpha-sphere
vertices -- the Voronoi vertices it fits to approximate the empty cavity's
shape, rather than the protein atoms lining it. Rendering them directly (one
`py3Dmol.addSphere` per vertex, at fpocket's own radius, semi-transparent) gives
a "filled space" volume for each pocket instead of a stick skeleton -- alongside
the cartoon backbone, same as before.


In [ ]:
import itertools

import py3Dmol
from IPython.display import HTML, display

# 3Dmol.js only defines "*Carbon" colorscheme presets for these 8 colors, but
# addSphere's "color" takes a plain color name, so the bare names work directly.
_POCKET_COLORS = ["orange", "cyan", "magenta", "yellow", "green", "purple", "blue", "white"]

druggable_pockets = [p for p in pockets if p["druggability_score"] is not None and p["druggability_score"] >= 0.1]
print(f"{len(druggable_pockets)} pocket(s) with druggability_score >= 0.2")

with open(os.path.join("af_data", pdb_files[0])) as f:
    pdb_text = f.read()

view = py3Dmol.view(width=700, height=550)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.8}})

legend_html = ""
for pocket, color in zip(druggable_pockets, itertools.cycle(_POCKET_COLORS)):
    for sphere in pocket["spheres"]:
        view.addSphere(
            {
                "center": {"x": sphere["x"], "y": sphere["y"], "z": sphere["z"]},
                "radius": sphere["radius"],
                "color": color,
                "opacity": 0.5,
            }
        )
    legend_html += (
        f'<div><span style="display:inline-block;width:12px;height:12px;background:{color};border:1px solid #888;margin-right:6px;"></span>'
        f'pocket {pocket["pocket_id"]} (druggability {pocket["druggability_score"]:.3f}, '
        f'{len(pocket["spheres"])} spheres)</div>'
    )
view.zoomTo()

display(HTML(f'<div style="display:flex; gap:16px;">'
             f'<div id="chem-pockets-volume-view" style="width:700px; height:550px; border:1px solid #ccc;"></div>'
             f'<div>{legend_html}</div></div>'))
view.insert("chem-pockets-volume-view")


# RCSB PDB structures for thrombin (UniProt entry name lookup)

`chem.rcsb.download_structures` accepts the same three id forms as
`chem.chembl.download_activities` (ChEMBL target id, UniProt accession, or
UniProt entry name). It resolves the id to a UniProt accession, finds every
PDB entry annotated with that accession via the RCSB Search API, optionally
filters by resolution (entries without one, e.g. NMR structures, are dropped
whenever a threshold is given), and downloads each entry's structure file(s).
Files already present in `outdir` are left alone, so re-running only fetches
what's missing.

In [ ]:
from chem import rcsb

n = rcsb.download_structures(
    "THRB_HUMAN",
    resolution_thres=2.0,
    filetype="pdb",
    outdir="data",
)
n

## Aligning multiple thrombin structures with chem.protein.align

`chem.protein.align` sequence-aligns and structurally superposes a set of
same-target structures (PDB/CIF, RCSB/AlphaFold freely mixed) onto a
reference, selecting each structure's primary polymer chain automatically
(the one with the most standard amino acid residues -- e.g. thrombin's
catalytic heavy chain rather than its short light chain). Here every
downloaded RCSB structure is aligned onto the AlphaFold model (fixed as the
reference, since it doesn't need to be a member of `structures`). It writes
one PDB file per input into `outdir`, so each can be overlaid in the same
viewer.

In [ ]:
import glob
import os

import pandas as pd

from chem import protein

# Reference is fixed to the AlphaFold model -- not user-selectable.
reference_path = glob.glob("af_data/*.pdb")[0]
reference_id = os.path.splitext(os.path.basename(reference_path))[0]

structures = sorted(glob.glob("data/*.pdb"))
structure_ids = [os.path.splitext(os.path.basename(p))[0] for p in structures]
id_to_path = dict(zip(structure_ids, structures))

# Align every downloaded RCSB structure onto the fixed AlphaFold reference, once.
# align() returns both rmsd (Angstroms) and identity (fraction of sequence-matched,
# gap-free residues that are identical -- gapped/unmatched residues aren't counted
# in either the numerator or the denominator), both plain floats rounded to 3dp.
align_results = protein.align(structures, reference=reference_path, outdir="aligned")
align_df = pd.DataFrame(
    [
        {"id": os.path.splitext(os.path.basename(p))[0], "rmsd": r["rmsd"], "identity": r["identity"]}
        for p, r in align_results.items()
    ]
).sort_values("rmsd")
# .style.hide(axis="index") drops the index column and, unlike the plain
# DataFrame repr, never truncates columns/rows with "..." regardless of count.
display(align_df.style.hide(axis="index").format({"rmsd": "{:.3f}", "identity": "{:.1%}"}))


## Overlaying the aligned structures

The reference is fixed to the AlphaFold model, and every downloaded RCSB
structure is aligned onto it up front (the RMSD/identity table above). Toggle
any number of the resulting structures on to overlay them in the viewer --
independent `ToggleButton`s, so several can be selected at once, updating the
view immediately (no realignment needed, since `aligned/` already has every
structure).


In [ ]:
import itertools
import os
import uuid

import ipywidgets as widgets
import py3Dmol
from IPython.display import HTML, display

from chem.protein import WATER

# Only water is excluded from display -- ions, crystallization additives,
# glycosylation sugars, modified residues, etc. are all shown as sticks, since
# they're real parts of the structure even when they aren't "the" ligand.
_display_exclude = WATER

# One independently-toggleable checkbox-style button per structure (unlike
# ToggleButtons, several can be selected at once) for choosing which of the
# already-aligned structures to overlay in the viewer below.
_default_overlay = set(structure_ids[:4])
overlay_toggles = {
    i: widgets.ToggleButton(description=i, value=(i in _default_overlay), layout=widgets.Layout(width="65px"))
    for i in structure_ids
}
overlay_box = widgets.GridBox(
    list(overlay_toggles.values()),
    layout=widgets.Layout(grid_template_columns="repeat(auto-fill, 65px)", width="100%"),
)

# 3Dmol.js only defines "*Carbon" colorscheme presets for these 8 colors.
_color_cycle = itertools.cycle(["orange", "cyan", "magenta", "yellow", "green", "purple", "blue", "white"])

view = py3Dmol.view(width=600, height=450)
_pdb_text = {}  # struct_id -> file contents, cached once per structure
_model_index = {}  # struct_id -> 3Dmol model index, assigned once on first load
_model_color = {}  # struct_id -> its permanently-assigned color (stable across toggles)


def _style_model(struct_id, visible):
    # An empty style hides a model without removing it, so model indices (and
    # everything else already loaded into the live viewer) stay stable -- this
    # is what lets a toggle only ever *update* the existing viewer instance
    # (view.update(), below) instead of tearing it down and rebuilding it from
    # scratch, which is what would reset the camera to the default framing on
    # every click and throw away any manual rotate/zoom/pan the user just did.
    idx = _model_index[struct_id]
    if not visible:
        view.setStyle({"model": idx}, {})
        return
    color = _model_color[struct_id]
    view.setStyle({"model": idx}, {"line": {"color": color}})
    # Line only draws the polymer backbone, so ligands need an explicit style.
    ligand_resnames = sorted(
        {line[17:20].strip() for line in _pdb_text[struct_id].splitlines() if line.startswith("HETATM")}
        - _display_exclude
    )
    if ligand_resnames:
        view.addStyle({"model": idx, "resn": ligand_resnames}, {"stick": {"colorscheme": f"{color}Carbon"}})


def _load_model(struct_id):
    aligned_path = os.path.join("aligned", f"{struct_id}.pdb")
    with open(aligned_path) as f:
        _pdb_text[struct_id] = f.read()
    view.addModel(_pdb_text[struct_id], "pdb")
    _model_index[struct_id] = len(_model_index)
    _model_color[struct_id] = next(_color_cycle)
    _style_model(struct_id, visible=True)


# A small dedicated sink for the incremental view.update() script tags a toggle
# emits, cleared each time -- kept separate from the viewer's own frame_id div
# (below) so re-styling never touches that div and never re-triggers insert().
_update_sink = widgets.Output()


def render_overlay(*_):
    chosen = {i for i, t in overlay_toggles.items() if t.value}
    for struct_id in structure_ids:
        if struct_id in chosen:
            if struct_id not in _model_index:
                _load_model(struct_id)
            else:
                _style_model(struct_id, visible=True)
        elif struct_id in _model_index:
            _style_model(struct_id, visible=False)
    _update_sink.clear_output(wait=True)
    with _update_sink:
        view.update()


# Reference is always loaded and visible; it's never one of the toggles.
_load_model(reference_id)
for struct_id in structure_ids:
    if struct_id in _default_overlay:
        _load_model(struct_id)
# zoomTo() only ever runs here, at setup -- never inside render_overlay -- so
# toggling a structure on/off never moves the camera.
view.zoomTo({"model": _model_index[reference_id]})

frame_id = f"chem-overlay-{uuid.uuid4().hex}"
display(HTML(f'<div id="{frame_id}" style="width:600px; height:450px; border:1px solid #ccc;"></div>'))
view.insert(frame_id)

for _toggle in overlay_toggles.values():
    _toggle.observe(render_overlay, names="value")

display(widgets.VBox([overlay_box, _update_sink]))


## Active site by ligand-position consensus

Every RCSB structure is already superposed onto the AlphaFold model's frame
(the `aligned/` step above), and each one carries its own bound ligand --
the most direct evidence there is for where the active site sits, no
separate pocket-detection tool (fpocket) needed. For every aligned structure
with a usable ligand (the largest HETATM group excluding solvent/ions and
the glycosylation sugar `NAG` / sulfotyrosine `TYS` / crystallization
additive `MRD`), every AlphaFold-model residue with at least one atom within
`dist_thres` of any of that ligand's atoms counts as a hit for that
structure. `threshold` is the minimum fraction of ligand-bearing structures
a residue must be a hit in to count as "active site" -- residues near many
different bound ligands are core to the site; residues near only one or two
are more likely a single crystal's own peripheral or crystallization-specific
contact. The distance search and every residue reported are computed
directly on the AlphaFold model -- the RCSB structures only ever contribute
their ligand coordinates.

`ligand_active_site_resnums` (AlphaFold numbering, recomputed live as either
slider moves) is the definitive list to export downstream.


In [ ]:
import os

import ipywidgets as widgets
import numpy as np
import pandas as pd
import py3Dmol
from Bio.PDB import NeighborSearch
from IPython.display import HTML, display

from chem.protein import SOLVENT_AND_IONS
from chem.protein.structural_align import _load_structure, _select_chain

# NAG (glycosylation) and TYS (sulfated hirudin-fragment tyrosine) aren't solvent,
# but they aren't the pharmacological ligand either -- exclude them too.
_ligand_exclude = SOLVENT_AND_IONS | {"NAG", "TYS", "MRD"}


def _pick_ligand_atoms(path, exclude):
    """(resname, Nx3 coord array) for the largest non-excluded HETATM group in a
    PDB file, or (None, None) if it has none. Raw-line parsing (not Bio.PDB) is
    enough here -- only the coordinates are needed, not a full structure object.
    """
    groups = {}  # resname -> list of (x, y, z)
    with open(path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            resname = line[17:20].strip()
            if resname in exclude:
                continue
            groups.setdefault(resname, []).append(
                (float(line[30:38]), float(line[38:46]), float(line[46:54]))
            )
    if not groups:
        return None, None
    resname = max(groups, key=lambda k: len(groups[k]))
    return resname, np.array(groups[resname])


# AlphaFold model chain -- the search space and reported resnums are always this
# structure, never any individual RCSB crystal form.
_af_structure = _load_structure(os.path.join("aligned", f"{reference_id}.pdb"))
_af_chain = _select_chain(next(_af_structure.get_models()))
_af_atoms = [a for r in _af_chain if r.id[0] == " " for a in r.get_atoms()]
_af_ns = NeighborSearch(_af_atoms)
_af_resname = {r.id[1]: r.get_resname() for r in _af_chain if r.id[0] == " "}

with open(os.path.join("aligned", f"{reference_id}.pdb")) as f:
    _af_pdb_text = f.read()

dist_slider = widgets.FloatSlider(
    value=5.0, min=1.0, max=10.0, step=0.5, description="Dist (Å):", readout_format=".1f", continuous_update=False
)
threshold_slider = widgets.FloatSlider(
    value=0.5, min=0.05, max=1.0, step=0.05, description="Threshold:", readout_format=".2f", continuous_update=False
)
ligand_consensus_output = widgets.Output()

ligand_consensus_df = None
ligand_active_site_resnums = None


def render_ligand_consensus(*_):
    global ligand_consensus_df, ligand_active_site_resnums
    ligand_consensus_output.clear_output(wait=True)
    with ligand_consensus_output:
        counts = {}  # af_resnum -> {"resname": ..., "count": int}
        n_analyzed = 0
        skipped = []

        for struct_id in structure_ids:
            aligned_path = os.path.join("aligned", f"{struct_id}.pdb")
            _resname, coords = _pick_ligand_atoms(aligned_path, _ligand_exclude)
            if coords is None:
                skipped.append(struct_id)
                continue
            n_analyzed += 1

            nearby_resnums = set()
            for xyz in coords:
                for atom in _af_ns.search(xyz, dist_slider.value):
                    nearby_resnums.add(atom.get_parent().id[1])
            for resnum in nearby_resnums:
                entry = counts.setdefault(resnum, {"resname": _af_resname[resnum], "count": 0})
                entry["count"] += 1

        ligand_consensus_df = pd.DataFrame(
            [
                {
                    "resnum": resnum,
                    "resname": v["resname"],
                    "count": v["count"],
                    "fraction": round(v["count"] / n_analyzed, 3),
                }
                for resnum, v in counts.items()
            ]
        ).sort_values(["fraction", "resnum"], ascending=[False, True])

        active = ligand_consensus_df[ligand_consensus_df["fraction"] >= threshold_slider.value].sort_values("resnum")
        ligand_active_site_resnums = active["resnum"].tolist()
        labels = [f"{r.resname.capitalize()}{r.resnum}" for r in active.itertuples()]

        print(f"{n_analyzed} structure(s) with a usable ligand, {len(skipped)} skipped (no non-solvent ligand)")
        print(
            f"{len(labels)} active site residues (ligand within {dist_slider.value:.1f}Å in "
            f">={threshold_slider.value * 100:.0f}% of {n_analyzed} structures):"
        )
        print(", ".join(labels) if labels else "(none)")

        view = py3Dmol.view(width=600, height=450)
        view.addModel(_af_pdb_text, "pdb")
        view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.5}})
        if ligand_active_site_resnums:
            view.addStyle({"resi": ligand_active_site_resnums}, {"stick": {"colorscheme": "greenCarbon"}})
            view.addResLabels(
                {"resi": ligand_active_site_resnums},
                {
                    "font": "sans-serif",
                    "fontSize": 12,
                    "fontColor": "black",
                    "showBackground": True,
                    "backgroundColor": "white",
                    "backgroundOpacity": 0.7,
                },
            )
        view.zoomTo({"resi": ligand_active_site_resnums} if ligand_active_site_resnums else {})
        view.show()


dist_slider.observe(render_ligand_consensus, names="value")
threshold_slider.observe(render_ligand_consensus, names="value")
render_ligand_consensus()

display(widgets.VBox([dist_slider, threshold_slider, ligand_consensus_output]))


### Active site residue table

`fraction` is the share of ligand-bearing structures whose ligand landed
within `dist_thres` of that residue, at the current slider position.
`resnum` is the AlphaFold model's own numbering, matching the viewer above.


In [ ]:
active_site_df = ligand_consensus_df[ligand_consensus_df["fraction"] >= threshold_slider.value].sort_values(
    "resnum"
).reset_index(drop=True)
display(active_site_df.style.hide(axis="index"))
